# 02 · Recommendation MovieLens —— 不是猜你喜欢，是你为什么会喜欢

**家族位置**：09 领域应用第 2 站。LR/FM 是传统推荐基线，DeepFM 自动学二阶交叉，DIN 用候选电影对用户历史行为做 attention 加权。

**学习目标**：隐式正负样本；FM 二阶交叉；DeepFM 的 DNN；DIN 行为序列 attention；AUC 同台对照。

## 1. 原理：从打分表到会看历史

### 通俗理解

**一句话**：LR 像把用户/电影各打一个分；FM 像算“这个人×这部片”的二人关系；DeepFM 再让小脑袋自动学组合；DIN 像翻你历史片单，看到候选电影和哪段经历最像就加权。

### 结构账

```
数据： MovieLens-100K u1.base/u1.test；rating≥4 正样本；每个正样本采3个未看负样本
切分：官方 u1.test 做 test；u1.base 每用户最后一条留 val，防时间泄漏
模型：LR（一次项）/ FM（二阶）/ DeepFM（FM+DNN）/ DIN（候选-历史 attention）
口径：BCEWithLogitsLoss，AUC；user/item embedding dim=16，5ep
```

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import load_movielens
from common.models import LRModel,FMModel,DeepFMModel,DINModel
from common.engine import train_rec,auc,make_history
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
root=Path('data/ml-100k')
train,val,test,genre,sizes=load_movielens(root,seed=0,negatives=3)
history=make_history(train)
print(f'sizes={sizes} genre={genre.shape} rows train/val/test={len(train)}/{len(val)}/{len(test)}')
print('正样本率 train/val/test:',*[round(float(x.label.mean()),3) for x in [train,val,test]])

## 2. 四模型训练：LR / FM / DeepFM / DIN

In [ ]:
models={'LR':LRModel(**sizes),'FM':FMModel(**sizes),'DeepFM':DeepFMModel(**sizes),'DIN':DINModel(**sizes)}
for name,m in models.items():
    print(f'\n{name} params={count_params(m)}',flush=True)
    train_rec(m,train,genre,history=history if name=='DIN' else None,epochs=5,lr=1e-3)
    print(f'{name} val-AUC={auc(m,val,genre,history if name=="DIN" else None):.4f} test-AUC={auc(m,test,genre,history if name=="DIN" else None):.4f}',flush=True)

## 3. AUC 对照 + DIN attention 可解释性

In [ ]:
res={n:auc(m,test,genre,history if n=='DIN' else None) for n,m in models.items()}
print('AUC summary:',{k:round(v,4) for k,v in res.items()},flush=True)
fig,ax=plt.subplots(figsize=(6,3.2))
ax.bar(list(res),list(res.values()),color=['#4C72B0','#55A868','#C44E52','#DD8452'])
for i,v in enumerate(res.values()): ax.text(i,v+0.005,f'{v:.4f}',ha='center')
ax.set_ylim(min(res.values())-0.03,1.0); ax.set_ylabel('test AUC'); ax.set_title('MovieLens-100K：传统→深度推荐对照')
plt.tight_layout(); plt.savefig(FIGS/'fig1_auc.png',dpi=150,bbox_inches='tight'); plt.show()
# DIN 历史长度消融：真实历史 vs 只给候选自身（无历史信息）
din=models['DIN']; base=auc(din,test,genre,history=None)
print(f'DIN full-history AUC={res["DIN"]:.4f} | no-history AUC={base:.4f} | gain={res["DIN"]-base:+.4f}',flush=True)
fig,ax=plt.subplots(figsize=(5,3)); ax.bar(['no history','full history'],[base,res['DIN']],color=['#AAAAAA','#DD8452'])
for i,v in enumerate([base,res['DIN']]): ax.text(i,v+0.005,f'{v:.4f}',ha='center')
ax.set_ylim(min(base,res['DIN'])-0.03,1.0); ax.set_title('DIN：用户历史 attention 是否有用')
plt.tight_layout(); plt.savefig(FIGS/'fig2_din.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',[(n,count_params(m),round(res[n],4)) for n,m in models.items()])